# Music Annotation Project V8.6 - Notebook 46R6
## Fail-closed, page-checkpointed, resumable mask x restorer crossover with whole-page candidates

**Status:** development-only scientific audit. This notebook has no production or promotion authority.

46R6 carries forward the 46R5 experiment while tightening resumability and auditability. It retains the frozen M0-M4 mask arms, the pre-registered restorer arms, source-group held-out learned restoration, exact outside-mask immutability, whole-page candidate assembly, post-output truth scoring, paired page-cluster inference and a curated final upload bundle.

### 46R6 hardening
- Run identity is explicit and versioned as 46R6.
- Existing page checkpoints are reusable only when their recorded contract fingerprint matches the current run.
- Partial or corrupt checkpoints fail closed and are recomputed, never silently accepted.
- Final artefact packaging includes notebook identity and checkpoint-audit records.
- All code cells are syntax-checked before release of this notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow scikit-image matplotlib torch torchvision

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, random, shutil, sys, platform, zipfile, gc, time, os
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from skimage.restoration import inpaint as sk_inpaint
from skimage.metrics import structural_similarity as ssim
from torch import nn
from torch.utils.data import Dataset, DataLoader

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
N45_ROOT = ROOT/'observed_print_classifier_v8_6_notebook45'
OUTROOT = ROOT/'observed_print_classifier_v8_6_notebook46'

SOURCE_N45_RUN_OVERRIDE = None
RESUME_RUN_OVERRIDE = None  # Strongly recommended: paste the exact crashed run directory here.
AUTO_RESUME_LATEST_INCOMPLETE = True
ALLOW_NEW_RUN_IF_NONE_FOUND = True

SEED = 46001
BOOT = 10000
PATCH = 128
EPOCHS = 8
BATCH = 16
ENABLE_LEARNED_RESTORER = True
VALIDATE_EXISTING_OUTPUT_SHA = False  # Shape and immutability are always checked; SHA is recorded.
VALIDATE_EXISTING_CANDIDATE_SHA = False  # Optional expensive hash check against prior candidate inventory.
REUSE_VALID_WHOLE_PAGE_CANDIDATES = True
SAVE_PROGRESS_EVERY_REGION = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if ENABLE_LEARNED_RESTORER and DEVICE != 'cuda':
    raise RuntimeError('Enable a GPU or set ENABLE_LEARNED_RESTORER=False.')


def candidate_resume_runs(root):
    required = ['source_record.json', 'restorer_pre_registration.json']
    hits = []
    if root.exists():
        for p in root.iterdir():
            if p.is_dir() and all((p/r).exists() for r in required):
                # A completed run is still safe to reopen, but incomplete runs are preferred.
                complete = (p/'v8_6_verdict.json').exists() and (p/'tables/paired_region_metrics.csv').exists()
                hits.append((complete, p.stat().st_mtime, p))
    return [p for complete, _, p in sorted(hits, key=lambda x:(x[0], -x[1]))]

if RESUME_RUN_OVERRIDE is not None:
    OUT = Path(RESUME_RUN_OVERRIDE)
    if not OUT.exists():
        raise RuntimeError(f'RESUME_RUN_OVERRIDE does not exist: {OUT}')
    RESUME_MODE = True
elif AUTO_RESUME_LATEST_INCOMPLETE:
    hits = candidate_resume_runs(OUTROOT)
    OUT = hits[0] if hits else None
    RESUME_MODE = OUT is not None
else:
    OUT = None
    RESUME_MODE = False

if OUT is None:
    if not ALLOW_NEW_RUN_IF_NONE_FOUND:
        raise RuntimeError('No resumable Notebook 46 run found. Set RESUME_RUN_OVERRIDE.')
    run = datetime.now().strftime('%Y%m%d_%H%M%S') + '_v8_6_notebook46R'
    OUT = OUTROOT/run
    RESUME_MODE = False

TABLES=OUT/'tables'; OUTPUTS=OUT/'outputs'; MODELS=OUT/'models'; REVIEW=OUT/'visual_audit'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'; WHOLE=OUT/'whole_page_candidates'
for p in [OUT,TABLES,OUTPUTS,MODELS,REVIEW,AUDIT,UPLOAD,WHOLE]:
    p.mkdir(parents=True, exist_ok=True)

print('Mode:', 'RESUME' if RESUME_MODE else 'NEW RUN')
print('Output:', OUT)
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU')

In [ ]:
# 46R6 contract fingerprint and atomic I/O helpers.
NOTEBOOK_VERSION = "46R6"
CONTRACT_FIELDS = {
    "notebook_version": NOTEBOOK_VERSION,
    "programme": "V8.6",
    "mask_arms": ["M0", "M1", "M2", "M3", "M4"],
    "development_only": True,
    "outside_mask_immutable": True,
    "truth_scoring_post_output_only": True,
}

def canonical_json_bytes(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode("utf-8")

def contract_fingerprint(extra=None):
    payload = dict(CONTRACT_FIELDS)
    if extra:
        payload["runtime"] = extra
    return hashlib.sha256(canonical_json_bytes(payload)).hexdigest()

def atomic_write_bytes(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "wb") as f:
        f.write(payload)
        f.flush()
        os.fsync(f.fileno())
    tmp.replace(path)

def atomic_write_json(path, obj):
    atomic_write_bytes(path, json.dumps(obj, indent=2, sort_keys=True).encode("utf-8"))

def validate_checkpoint_record(record, expected_fingerprint, required_paths=()):
    if not isinstance(record, dict):
        return False, "record_not_dict"
    if record.get("contract_fingerprint") != expected_fingerprint:
        return False, "contract_fingerprint_mismatch"
    if record.get("status") != "complete":
        return False, "checkpoint_not_complete"
    for p in required_paths:
        if not Path(p).exists():
            return False, f"missing:{p}"
    return True, "valid"

RUN_CONTRACT_FINGERPRINT = contract_fingerprint()
print("46R6 contract fingerprint:", RUN_CONTRACT_FINGERPRINT)


In [ ]:
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):
            h.update(b)
    return h.hexdigest()

def atomic_json(path, value):
    path=Path(path); tmp=Path(str(path)+'.tmp')
    tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str))
    tmp.replace(path)

def atomic_csv(frame, path):
    path=Path(path); tmp=Path(str(path)+'.tmp')
    frame.to_csv(tmp,index=False); tmp.replace(path)

def latest(root,req):
    hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in req)] if Path(root).exists() else []
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

REQ45=['mask_lock.json','notebook45_verdict.json','source_record.json','tables/region_manifest.parquet','tables/mask_readiness_metrics.csv']
N45=Path(SOURCE_N45_RUN_OVERRIDE) if SOURCE_N45_RUN_OVERRIDE else latest(N45_ROOT,REQ45)
if N45 is None:
    raise RuntimeError('No complete Notebook 45 run found.')

lock=json.loads((N45/'mask_lock.json').read_text())
verdict45=json.loads((N45/'notebook45_verdict.json').read_text())
source=json.loads((N45/'source_record.json').read_text())
regions=pd.read_parquet(N45/'tables/region_manifest.parquet')

record={
    'notebook45':str(N45),
    'mask_lock_sha256':sha(N45/'mask_lock.json'),
    'mask_lock_id':lock['lock_id'],
    'source_record_sha256':sha(N45/'source_record.json'),
    'notebook45_ready':verdict45.get('ready_for_crossover'),
}

source_record_path=OUT/'source_record.json'
if source_record_path.exists():
    previous=json.loads(source_record_path.read_text())
    for key in ['mask_lock_sha256','mask_lock_id','source_record_sha256']:
        if previous.get(key)!=record.get(key):
            raise RuntimeError(f'Resume blocked: {key} differs from the crashed run.')
else:
    atomic_json(source_record_path,record)

# Verify every region has five readable, full-page Notebook 45 masks.
for _,r in regions.iterrows():
    for arm in ['M0','M1','M2','M3','M4']:
        p=N45/'masks'/r.page/r.region_id.replace(':','_')/f'{arm}.png'
        if not p.exists(): raise RuntimeError(f'Missing Notebook 45 mask: {p}')

print(json.dumps(record,indent=2))
print('Regions:',len(regions),'Pages:',regions.page.nunique(),'Source groups:',regions.source_group.nunique())

In [ ]:
RESTORERS=['R0_unresolved','R1_biharmonic','R2_telea']+(['R3_learned_unet'] if ENABLE_LEARNED_RESTORER else [])
PRE={
    'version':'8.6-notebook46R-resumable-development',
    'mask_lock_id':lock['lock_id'],
    'restorers':RESTORERS,
    'epochs':EPOCHS,
    'patch':PATCH,
    'seed':SEED,
    'clean_truth_role':'training targets on permitted source groups and post-output scoring only',
    'best_of_n_selection':False,
    'resume_contract':{
        'reuse_valid_checkpoints':True,
        'reuse_valid_outputs':True,
        'recompute_missing_or_invalid_only':True,
        'progress_written_after_each_region':True,
    },
}
pre_path=OUT/'restorer_pre_registration.json'
if pre_path.exists():
    old=json.loads(pre_path.read_text())
    immutable_keys=['mask_lock_id','restorers','epochs','patch','seed']
    for key in immutable_keys:
        if old.get(key)!=PRE.get(key):
            raise RuntimeError(f'Resume blocked: pre-registration field changed: {key}')
else:
    atomic_json(pre_path,PRE)

source_v82=Path(source['source_v82'])
clean_map={p.stem:p for p in (ROOT/'source'/'clean').iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}}
page_obs={p:cv2.imread(str(source_v82/'whole_pages'/p/'observed.png'),0) for p in regions.page.unique()}
page_clean={p:cv2.imread(str(clean_map[p]),0) for p in regions.page.unique()}
for p in page_obs:
    if page_obs[p] is None or page_clean[p] is None or page_obs[p].shape!=page_clean[p].shape:
        raise RuntimeError(f'{p}: page mismatch')
print('Pre-registration verified. Existing run may be resumed.')

In [ ]:
def reconstruct_classical(obs,mask,method):
 m=(mask>0).astype(np.uint8)*255
 if not m.any():return obs.copy()
 if method=='R0_unresolved':return obs.copy()
 if method=='R2_telea':cand=cv2.inpaint(obs,m,3,cv2.INPAINT_TELEA)
 elif method=='R1_biharmonic':cand=np.clip(sk_inpaint.inpaint_biharmonic(obs.astype(np.float32)/255.,m>0,channel_axis=None)*255,0,255).astype(np.uint8)
 out=obs.copy();out[m>0]=cand[m>0];assert np.array_equal(out[m==0],obs[m==0]);return out

## Learned deterministic restorer: fold-trained synthetic masked U-Net

In [ ]:
class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1 = nn.Sequential(
            nn.Conv2d(2, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
        )
        self.p = nn.MaxPool2d(2)
        self.e2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
        )
        self.u = nn.ConvTranspose2d(64, 32, 2, 2)
        self.d = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        a = self.e1(x)
        b = self.e2(self.p(a))
        return self.d(torch.cat([self.u(b), a], 1))


def centred_patch_geometry(shape, y0, y1, x0, x1, size=PATCH):
    """Return page and patch slices for one fixed-size patch centred on a region."""
    h, w = map(int, shape[:2])
    cy = (int(y0) + int(y1)) // 2
    cx = (int(x0) + int(x1)) // 2
    half = size // 2

    nominal_y0 = cy - half
    nominal_x0 = cx - half
    nominal_y1 = nominal_y0 + size
    nominal_x1 = nominal_x0 + size

    page_y0 = max(0, nominal_y0)
    page_x0 = max(0, nominal_x0)
    page_y1 = min(h, nominal_y1)
    page_x1 = min(w, nominal_x1)

    patch_y0 = page_y0 - nominal_y0
    patch_x0 = page_x0 - nominal_x0
    patch_y1 = patch_y0 + (page_y1 - page_y0)
    patch_x1 = patch_x0 + (page_x1 - page_x0)

    if not (0 <= page_y0 <= page_y1 <= h and 0 <= page_x0 <= page_x1 <= w):
        raise RuntimeError(
            f'Invalid page slice for shape={shape}, region={(y0, y1, x0, x1)}'
        )
    if not (0 <= patch_y0 <= patch_y1 <= size and 0 <= patch_x0 <= patch_x1 <= size):
        raise RuntimeError(
            f'Invalid patch slice for shape={shape}, region={(y0, y1, x0, x1)}'
        )

    return {
        'page': (slice(page_y0, page_y1), slice(page_x0, page_x1)),
        'patch': (slice(patch_y0, patch_y1), slice(patch_x0, patch_x1)),
        'centre': (cy, cx),
        'nominal_origin': (nominal_y0, nominal_x0),
    }


def crop_pad(a, y0, y1, x0, x1, size=PATCH, pad_mode='reflect'):
    """Always return exactly size x size, with the region centre spatially aligned."""
    if a.ndim != 2:
        raise ValueError(f'crop_pad expects a 2D array, got shape {a.shape}')

    geom = centred_patch_geometry(a.shape, y0, y1, x0, x1, size)
    out = np.empty((size, size), dtype=a.dtype)

    # Safe constant initialisation before the real page area is copied.
    fill = 0 if a.dtype == bool else (255 if np.issubdtype(a.dtype, np.integer) else 0.0)
    out[...] = fill
    out[geom['patch']] = a[geom['page']]

    # Fill off-page areas using actual padding where possible. Reflect padding
    # requires dimensions greater than one; fall back to edge padding otherwise.
    py0, px0 = geom['patch'][0].start, geom['patch'][1].start
    py1, px1 = size - geom['patch'][0].stop, size - geom['patch'][1].stop
    if py0 or py1 or px0 or px1:
        core = a[geom['page']]
        if core.size == 0:
            raise RuntimeError(
                f'Empty crop core for shape={a.shape}, region={(y0, y1, x0, x1)}'
            )
        mode = pad_mode
        if mode == 'reflect' and (core.shape[0] < 2 or core.shape[1] < 2):
            mode = 'edge'
        out = np.pad(core, ((py0, py1), (px0, px1)), mode=mode)

    if out.shape != (size, size):
        raise RuntimeError(
            f'crop_pad produced {out.shape}, expected {(size, size)}; '
            f'input={a.shape}, region={(y0, y1, x0, x1)}'
        )
    return out


class SyntheticDS(Dataset):
    def __init__(self, rows, pages, n=3000):
        self.rows = rows.reset_index(drop=True)
        self.pages = list(pages)
        self.n = int(n)
        self.rng = np.random.default_rng(SEED)

        if self.rows.empty:
            raise RuntimeError('SyntheticDS received no residual regions.')
        if not self.pages:
            raise RuntimeError('SyntheticDS received no training pages.')

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        page = self.pages[i % len(self.pages)]
        clean = page_clean[page]

        # Draw a residual shape independently from the clean training page.
        r = self.rows.iloc[i % len(self.rows)]
        y0, y1, x0, x1 = map(int, [r.y0, r.y1, r.x0, r.x1])

        target = crop_pad(clean, y0, y1, x0, x1)

        mask_path = (
            N45 / 'masks' / str(r.page) /
            str(r.region_id).replace(':', '_') /
            f"M{2 + (i % 2)}.png"
        )
        mask_full = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask_full is None:
            raise RuntimeError(f'Cannot read training mask: {mask_path}')

        # The stored masks are full-page masks. Use the source region geometry
        # associated with that mask, then transplant the resulting shape into
        # the fixed-size training patch.
        mask = crop_pad(
            (mask_full > 0).astype(np.uint8),
            y0, y1, x0, x1,
            pad_mode='constant',
        ) > 0

        if target.shape != (PATCH, PATCH) or mask.shape != (PATCH, PATCH):
            raise RuntimeError(
                f'Training patch mismatch: target={target.shape}, mask={mask.shape}, '
                f'region={r.region_id}'
            )

        corrupted = target.copy()
        paper = int(np.median(target[target > 200])) if np.any(target > 200) else 255
        corrupted[mask] = paper

        X = np.stack([
            corrupted.astype(np.float32) / 255.0,
            mask.astype(np.float32),
        ]).astype(np.float32)
        Y = (target.astype(np.float32) / 255.0)[None]
        M = mask.astype(np.float32)[None]

        if X.shape != (2, PATCH, PATCH) or Y.shape != (1, PATCH, PATCH) or M.shape != (1, PATCH, PATCH):
            raise RuntimeError(
                f'Unexpected dataset tensor shapes: X={X.shape}, Y={Y.shape}, M={M.shape}'
            )

        return torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(M)


def train_model(train_pages, fold_name):
    fold_rows = regions[regions.page.isin(train_pages)].copy()
    ds = SyntheticDS(fold_rows, train_pages)
    dl = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=True,
        num_workers=2,
        pin_memory=(DEVICE == 'cuda'),
    )
    model = TinyUNet().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for ep in range(EPOCHS):
        model.train()
        total = 0.0
        for x, y, mask in dl:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            mask = mask.to(DEVICE, non_blocking=True)
            pred = model(x)
            loss = ((pred - y).abs() * mask).sum() / mask.sum().clamp_min(1)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            total += float(loss.detach().cpu())
        print(f'{fold_name} epoch {ep + 1}/{EPOCHS} loss {total / max(len(dl), 1):.6f}')

    path = MODELS / f'{fold_name}.pt'
    torch.save(model.state_dict(), path)
    return model, path

In [ ]:
# Outer source-group checkpoints. Valid checkpoints are reused; missing/corrupt ones are retrained.
models={}; model_rows=[]

def checkpoint_name(group):
    return f'leaveout_{hashlib.sha1(str(group).encode()).hexdigest()[:10]}.pt'

def load_checkpoint(path):
    model=TinyUNet().to(DEVICE)
    state=torch.load(path,map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()
    # Structural smoke test.
    with torch.no_grad():
        y=model(torch.zeros((1,2,PATCH,PATCH),dtype=torch.float32,device=DEVICE))
    if tuple(y.shape)!=(1,1,PATCH,PATCH) or not torch.isfinite(y).all():
        raise RuntimeError(f'Checkpoint smoke test failed: {path}')
    return model

if ENABLE_LEARNED_RESTORER:
    groups=sorted(regions.source_group.unique())
    for g in groups:
        fold_name=f'leaveout_{hashlib.sha1(str(g).encode()).hexdigest()[:10]}'
        path=MODELS/checkpoint_name(g)
        status='reused'
        try:
            if not path.exists(): raise FileNotFoundError(path)
            model=load_checkpoint(path)
        except Exception as error:
            status='rebuilt'
            if path.exists():
                quarantined=path.with_suffix(path.suffix+'.invalid')
                path.replace(quarantined)
                print('Quarantined invalid checkpoint:',quarantined,'reason:',error)
            train_pages=regions.loc[regions.source_group!=g,'page'].unique()
            if len(train_pages)==0: raise RuntimeError('No training pages outside source group')
            model,path=train_model(train_pages,fold_name)
            model=load_checkpoint(path)
        models[g]=model
        model_rows.append({
            'test_source_group':g,'checkpoint':str(path),'sha256':sha(path),
            'status':status,'train_pages':int(regions.loc[regions.source_group!=g,'page'].nunique()),
        })
        atomic_csv(pd.DataFrame(model_rows),AUDIT/'learned_model_inventory.csv')
        print(status.upper(),g,path.name)
else:
    atomic_csv(pd.DataFrame(columns=['test_source_group','checkpoint','sha256','status','train_pages']),AUDIT/'learned_model_inventory.csv')

In [ ]:
def run_learned(model, obs, mask, r):
    y0,y1,x0,x1=map(int,[r.y0,r.y1,r.x0,r.x1])
    crop=crop_pad(obs,y0,y1,x0,x1)
    mask_crop=crop_pad(mask.astype(np.uint8),y0,y1,x0,x1,pad_mode='constant')>0
    if crop.shape!=(PATCH,PATCH) or mask_crop.shape!=(PATCH,PATCH):
        raise RuntimeError(f'Inference patch mismatch for {r.region_id}: {crop.shape}, {mask_crop.shape}')
    paper=int(np.median(crop[crop>200])) if np.any(crop>200) else 255
    corrupted=crop.copy(); corrupted[mask_crop]=paper
    x=np.stack([corrupted.astype(np.float32)/255.0,mask_crop.astype(np.float32)]).astype(np.float32)
    model.eval()
    with torch.no_grad():
        pred=(model(torch.from_numpy(x)[None].to(DEVICE))[0,0].cpu().numpy()*255).clip(0,255).astype(np.uint8)
    if pred.shape!=(PATCH,PATCH):
        raise RuntimeError(f'Prediction shape mismatch: {pred.shape}')
    return pred


def place_learned_prediction(original,pred_patch,full_mask,r):
    geom=centred_patch_geometry(original.shape,int(r.y0),int(r.y1),int(r.x0),int(r.x1),PATCH)
    out=original.copy()
    page_mask=full_mask[geom['page']]
    local_pred=pred_patch[geom['patch']]
    local_out=out[geom['page']]
    if not (local_out.shape==page_mask.shape==local_pred.shape):
        raise RuntimeError(f'Placement mismatch {r.region_id}: {local_out.shape}, {page_mask.shape}, {local_pred.shape}')
    local_out[page_mask]=local_pred[page_mask]
    out[geom['page']]=local_out
    if not np.array_equal(out[~full_mask],original[~full_mask]):
        raise RuntimeError(f'{r.region_id}: outside-mask drift')
    return out


def expected_output_path(r,arm,rest):
    return OUTPUTS/r.page/r.region_id.replace(':','_')/f'{arm}_{rest}.png'


# R5 checkpoint contract.
# A valid completed-page checkpoint contains the complete regional output inventory
# for that page. On restart, one checkpoint-file read replaces thousands of output
# existence probes and all mask reads for that completed page.
PAGE_CHECKPOINTS=AUDIT/'page_completion'
PAGE_CHECKPOINTS.mkdir(parents=True,exist_ok=True)
ARMS=['M0','M1','M2','M3','M4']


def page_checkpoint_path(page):
    return PAGE_CHECKPOINTS/f'{page}.json'


def read_json_once(path):
    # One attempted file read. Missing/unreadable means incomplete and triggers recovery.
    try:
        return json.loads(Path(path).read_text())
    except (FileNotFoundError, OSError, json.JSONDecodeError):
        return None


def expected_page_rows(page_regions):
    return int(len(page_regions)*len(ARMS)*len(RESTORERS))


def load_completed_page_checkpoint(page,page_regions):
    cp=read_json_once(page_checkpoint_path(page))
    if cp is None:
        return None
    expected=expected_page_rows(page_regions)
    rows=cp.get('output_rows')
    valid=(
        cp.get('complete') is True
        and cp.get('page')==page
        and cp.get('mask_lock_id')==lock['lock_id']
        and cp.get('restorers')==RESTORERS
        and cp.get('arms')==ARMS
        and int(cp.get('expected_rows',-1))==expected
        and isinstance(rows,list)
        and len(rows)==expected
    )
    if not valid:
        return None
    keys={(str(x.get('region_id')),str(x.get('arm')),str(x.get('restorer'))) for x in rows}
    expected_keys={(str(r.region_id),arm,rest) for _,r in page_regions.iterrows() for arm in ARMS for rest in RESTORERS}
    if keys!=expected_keys:
        return None
    return rows


def write_completed_page_checkpoint(page,page_regions,page_rows):
    expected=expected_page_rows(page_regions)
    if len(page_rows)!=expected:
        raise RuntimeError(f'{page}: refusing incomplete page checkpoint {len(page_rows)} != {expected}')
    payload={
        'version':'46R6-page-checkpoint-v1',
        'page':page,
        'complete':True,
        'created_utc':datetime.now(timezone.utc).isoformat(),
        'mask_lock_id':lock['lock_id'],
        'arms':ARMS,
        'restorers':RESTORERS,
        'regions':int(len(page_regions)),
        'expected_rows':expected,
        'output_rows':page_rows,
    }
    atomic_json(page_checkpoint_path(page),payload)


def existing_output(path):
    # Deliberately existence-only. Decode/shape checks occur when an existing file
    # is consumed during final whole-page assembly.
    return path.exists()


def load_needed_masks(r,obs_shape,needed_arms):
    masks={}
    for arm in sorted(set(needed_arms)):
        p=N45/'masks'/r.page/r.region_id.replace(':','_')/f'{arm}.png'
        m=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE)
        if m is None or m.shape!=obs_shape:
            raise RuntimeError(f'Invalid Notebook 45 mask: {p}')
        masks[arm]=m>0
    return masks


# Single-pass R5 recovery/reconstruction.
# Completed page: one checkpoint read, zero regional output probes, zero mask reads.
# Incomplete page: test every expected output filename first; load only masks needed
# by genuinely missing outputs; checkpoint the page atomically after completion.
output_rows=[]
page_status=[]
for page,page_regions in regions.groupby('page',sort=False):
    checkpoint_rows=load_completed_page_checkpoint(page,page_regions)
    if checkpoint_rows is not None:
        output_rows.extend(checkpoint_rows)
        page_status.append({'page':page,'status':'checkpoint_reused','regions':len(page_regions),'rows':len(checkpoint_rows),'missing_outputs_found':0})
        print(f'SKIP COMPLETE PAGE {page}: checkpoint reused ({len(checkpoint_rows)} rows)')
        continue

    obs=page_obs[page]
    page_rows=[]
    missing_total=0

    for _,r in page_regions.iterrows():
        # First test all 20 expected filenames. No mask is loaded before this finishes.
        missing=[]
        for arm in ARMS:
            for rest in RESTORERS:
                path=expected_output_path(r,arm,rest)
                if existing_output(path):
                    page_rows.append({
                        'region_id':r.region_id,'page':page,'source_group':r.source_group,
                        'arm':arm,'restorer':rest,'output':str(path),'status':'reused'
                    })
                else:
                    missing.append((arm,rest,path))

        if not missing:
            continue

        missing_total+=len(missing)
        needed_arms=[arm for arm,_,_ in missing]
        masks=load_needed_masks(r,obs.shape,needed_arms)
        region_dir=OUTPUTS/page/r.region_id.replace(':','_')
        region_dir.mkdir(parents=True,exist_ok=True)

        for arm,rest,path in missing:
            mask=masks[arm]
            if rest=='R3_learned_unet':
                pred_patch=run_learned(models[r.source_group],obs,mask,r)
                out=place_learned_prediction(obs,pred_patch,mask,r)
            else:
                out=reconstruct_classical(obs,mask,rest)
            if out.shape!=obs.shape or not np.array_equal(out[~mask],obs[~mask]):
                raise RuntimeError(f'{r.region_id} {arm} {rest}: output contract failure')
            tmp=path.with_suffix('.tmp.png')
            if not cv2.imwrite(str(tmp),out):
                raise RuntimeError(f'Failed to write {tmp}')
            tmp.replace(path)
            page_rows.append({
                'region_id':r.region_id,'page':page,'source_group':r.source_group,
                'arm':arm,'restorer':rest,'output':str(path),'status':'computed'
            })

    # Exact page inventory check before the checkpoint becomes authoritative.
    expected=expected_page_rows(page_regions)
    page_df=pd.DataFrame(page_rows)
    if len(page_df)!=expected or page_df.duplicated(['region_id','arm','restorer']).any():
        raise RuntimeError(f'{page}: output inventory incomplete/duplicated: {len(page_df)} expected {expected}')
    expected_keys={(str(r.region_id),arm,rest) for _,r in page_regions.iterrows() for arm in ARMS for rest in RESTORERS}
    actual_keys={(str(x['region_id']),str(x['arm']),str(x['restorer'])) for x in page_rows}
    if actual_keys!=expected_keys:
        raise RuntimeError(f'{page}: output inventory key mismatch')

    write_completed_page_checkpoint(page,page_regions,page_rows)
    output_rows.extend(page_rows)
    page_status.append({'page':page,'status':'completed_and_checkpointed','regions':len(page_regions),'rows':len(page_rows),'missing_outputs_found':missing_total})

    # Durable progress is now page-granular rather than rewriting a growing CSV after every region.
    atomic_csv(pd.DataFrame(page_status),AUDIT/'page_checkpoint_status.csv')
    atomic_json(AUDIT/'resume_progress.json',{
        'version':'46R6',
        'updated_utc':datetime.now(timezone.utc).isoformat(),
        'last_completed_page':page,
        'completed_page_checkpoints':int(len(list(PAGE_CHECKPOINTS.glob('*.json')))),
        'rows_recorded_this_invocation':len(output_rows),
        'total_expected_rows':len(regions)*len(ARMS)*len(RESTORERS),
    })
    print(f'CHECKPOINTED PAGE {page}: regions={len(page_regions)}, rows={len(page_rows)}, newly computed={missing_total}')

outputs=pd.DataFrame(output_rows)
expected=len(regions)*len(ARMS)*len(RESTORERS)
if len(outputs)!=expected or outputs.duplicated(['region_id','arm','restorer']).any():
    raise RuntimeError(f'Output inventory incomplete/duplicated: {len(outputs)} expected {expected}')
atomic_csv(outputs,AUDIT/'output_inventory.csv')
atomic_csv(pd.DataFrame(page_status),AUDIT/'page_checkpoint_status.csv')
atomic_json(AUDIT/'output_completion.json',{
    'complete':True,
    'version':'46R6',
    'rows':len(outputs),
    'expected':expected,
    'completed_utc':datetime.now(timezone.utc).isoformat(),
    'checkpoint_reused_pages':int(sum(x['status']=='checkpoint_reused' for x in page_status)),
    'completed_pages_this_invocation':int(sum(x['status']=='completed_and_checkpointed' for x in page_status)),
    'computed_outputs_this_invocation':int((outputs.status=='computed').sum()),
    'reused_outputs_this_invocation':int((outputs.status=='reused').sum()),
    'completed_page_restart_cost':'one page-checkpoint JSON read; no regional output probes or mask reads',
})
print(outputs.status.value_counts())
print('RECONSTRUCTION INVENTORY COMPLETE')

In [ ]:
# Assemble proper whole-page candidates.
# R4 adds validated persistent reuse so resumed runs do not rebuild every whole-page candidate.
# Contract: start from frozen V7.8 (which removes confident annotation), restore
# exact observed bytes at Notebook 45 protected PRINT pixels, then replace only
# the selected residual arm with the corresponding restorer output.

source45=json.loads((N45/'source_record.json').read_text())
source_v82=Path(source45['source_v82'])
source_v84=Path(source45['source_v84'])
v75=Path(source45['v75'])

# Rebuild the protected-pixel map exactly as Notebook 45 did.
v75_status=pd.read_csv(v75/'tables/cache_status.csv')
v75_cache={str(r.page):Path(r.cache) for _,r in v75_status.iterrows()}
p82=pd.read_parquet(source_v82/'tables/all_eligible_oof_predictions.parquet')
oof84=pd.read_parquet(source_v84/'tables/04_oof_safety_and_value_predictions.parquet')
pix84=pd.read_parquet(source_v84/'tables/02_candidate_group_pixels.parquet')
if 'arm' in oof84.columns:
    oof84=oof84[oof84.arm=='multiscale_33_97'].copy()

page_protected={}; page_auth={}; page_base={}
for page in sorted(regions.page.unique()):
    if page not in v75_cache: raise RuntimeError(f'{page}: V7.5 cache missing for full-page assembly')
    with np.load(v75_cache[page]) as z:
        needed=['mask','g80_preserve']
        miss=[k for k in needed if k not in z.files]
        if miss: raise RuntimeError(f'{page}: V7.5 cache missing {miss}')
        auth=z['mask']>0; g78=z['g80_preserve']>0
    obs=page_obs[page]
    base=cv2.imread(str(source_v82/'whole_pages'/page/'v7_8.png'),cv2.IMREAD_GRAYSCALE)
    if base is None or base.shape!=obs.shape or auth.shape!=obs.shape:
        raise RuntimeError(f'{page}: observed/V7.8/authoritative-mask shape mismatch')
    if not np.array_equal(base[~auth],obs[~auth]):
        raise RuntimeError(f'{page}: frozen V7.8 differs from observed outside authoritative mask')

    pg=p82[p82.page.astype(str)==str(page)]
    strict=np.zeros(auth.shape,bool)
    if 'decision' in pg.columns:
        hit=pg.decision.astype(str).eq('PRINT').to_numpy()
    else:
        hit=pg.p_print.to_numpy(float)>=.995
    strict[pg.y.to_numpy(int)[hit],pg.x.to_numpy(int)[hit]]=True

    selected=set(oof84.loc[(oof84.page.astype(str)==str(page)) & oof84.selected.astype(bool),'group_id'].astype(str))
    q=pix84[(pix84.page.astype(str)==str(page)) & pix84.group_id.astype(str).isin(selected)]
    v84=np.zeros(auth.shape,bool)
    if len(q): v84[q.y.to_numpy(int),q.x.to_numpy(int)]=True
    protected=(g78|strict|v84)&auth
    page_protected[page]=protected; page_auth[page]=auth; page_base[page]=base

candidate_rows=[]
prior_candidate_inventory={}
prior_inventory_path=TABLES/'whole_page_candidate_inventory.csv'
if prior_inventory_path.exists():
    try:
        _pi=pd.read_csv(prior_inventory_path)
        if {'page','arm','restorer','candidate'}.issubset(_pi.columns):
            prior_candidate_inventory={(str(r.page),str(r.arm),str(r.restorer)):r.to_dict() for _,r in _pi.iterrows()}
    except Exception as e:
        print('Ignoring unreadable prior candidate inventory:',e)

reused_candidates=0; rebuilt_candidates=0

def validate_existing_candidate(path,obs,base,auth,protected,union,prior=None):
    if not REUSE_VALID_WHOLE_PAGE_CANDIDATES or not path.exists(): return None
    im=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
    if im is None or im.shape!=obs.shape:return None
    if np.any(im[~auth]!=obs[~auth]):return None
    if np.any(im[protected & ~union]!=obs[protected & ~union]):return None
    if np.any(im[(~union)&(~protected)]!=base[(~union)&(~protected)]):return None
    if prior and VALIDATE_EXISTING_CANDIDATE_SHA and str(prior.get('sha256','')) and sha(path)!=str(prior['sha256']):return None
    return im
for page in sorted(regions.page.unique()):
    obs=page_obs[page]; base=page_base[page]; auth=page_auth[page]; protected=page_protected[page]
    page_dir=WHOLE/page; page_dir.mkdir(parents=True,exist_ok=True)
    cv2.imwrite(str(page_dir/'observed.png'),obs)
    cv2.imwrite(str(page_dir/'v7_8.png'),base)
    cv2.imwrite(str(page_dir/'protected_print.png'),protected.astype(np.uint8)*255)
    page_regions=regions[regions.page==page]

    for arm in ['M0','M1','M2','M3','M4']:
        union=np.zeros(auth.shape,bool)
        region_masks={}
        for _,r in page_regions.iterrows():
            mp=N45/'masks'/page/r.region_id.replace(':','_')/f'{arm}.png'
            m=cv2.imread(str(mp),cv2.IMREAD_GRAYSCALE)
            if m is None or m.shape!=auth.shape: raise RuntimeError(f'Invalid assembly mask: {mp}')
            region_masks[r.region_id]=m>0; union|=m>0
        cv2.imwrite(str(page_dir/f'{arm}_residual_union.png'),union.astype(np.uint8)*255)

        for rest in RESTORERS:
            path=page_dir/f'{arm}_{rest}.png'
            prior=prior_candidate_inventory.get((str(page),str(arm),str(rest)))
            candidate=validate_existing_candidate(path,obs,base,auth,protected,union,prior)
            if candidate is not None:
                reused_candidates+=1
            else:
                candidate=base.copy()
                candidate[protected]=obs[protected]
                for _,r in page_regions.iterrows():
                    m=region_masks[r.region_id]
                    op=expected_output_path(r,arm,rest)
                    reconstructed=cv2.imread(str(op),cv2.IMREAD_GRAYSCALE)
                    if reconstructed is None or reconstructed.shape!=candidate.shape:
                        raise RuntimeError(f'Invalid regional reconstruction during assembly: {op}')
                    candidate[m]=reconstructed[m]
                tmp=path.with_suffix('.tmp.png')
                if not cv2.imwrite(str(tmp),candidate): raise RuntimeError(f'Cannot write {tmp}')
                tmp.replace(path)
                rebuilt_candidates+=1

            outside=int(np.sum(candidate[~auth]!=obs[~auth]))
            protected_errors=int(np.sum(candidate[protected & ~union]!=obs[protected & ~union]))
            if outside or protected_errors:
                raise RuntimeError(f'{page} {arm} {rest}: whole-page contract failed outside={outside}, protected={protected_errors}')
            candidate_rows.append({
                'page':page,'source_group':str(page_regions.source_group.iloc[0]),'arm':arm,'restorer':rest,
                'candidate':str(path),'sha256':sha(path),'authoritative_pixels':int(auth.sum()),
                'protected_print_pixels':int(protected.sum()),'residual_pixels':int(union.sum()),
                'changed_vs_observed':int(np.sum(candidate!=obs)),'changed_vs_v78':int(np.sum(candidate!=base)),
                'outside_mask_drift':outside,'protected_non_residual_errors':protected_errors,
            })

whole_candidates=pd.DataFrame(candidate_rows)
expected_candidates=regions.page.nunique()*5*len(RESTORERS)
if len(whole_candidates)!=expected_candidates or whole_candidates.duplicated(['page','arm','restorer']).any():
    raise RuntimeError(f'Whole-page candidate inventory incomplete: {len(whole_candidates)} expected {expected_candidates}')
atomic_csv(whole_candidates,TABLES/'whole_page_candidate_inventory.csv')
print('WHOLE-PAGE CANDIDATES COMPLETE:',len(whole_candidates))
print('Folder:',WHOLE)

## Post-output truth scoring

In [ ]:
def ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im):return cv2.Canny(im,70,170)>0
def f1(ref,pred,reg,tol=1):
    ref=ref&reg; pred=pred&reg
    if not ref.any():return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8)
    p=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1)
    q=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum()
    return p,q,2*p*q/max(p+q,1e-12)

inventory_path=AUDIT/'output_inventory.csv'
if not inventory_path.exists(): raise RuntimeError('Complete output inventory is required before truth scoring.')
outputs=pd.read_csv(inventory_path)
expected=len(regions)*5*len(RESTORERS)
if len(outputs)!=expected: raise RuntimeError(f'Incomplete output inventory: {len(outputs)} expected {expected}')

metric_path=TABLES/'paired_region_metrics.csv'
existing_metrics=pd.read_csv(metric_path) if metric_path.exists() else pd.DataFrame()
done=set(zip(existing_metrics.region_id,existing_metrics.arm,existing_metrics.restorer)) if len(existing_metrics) else set()
rows=existing_metrics.to_dict('records') if len(existing_metrics) else []
for _,o in outputs.iterrows():
    key=(o.region_id,o.arm,o.restorer)
    if key in done: continue
    r=regions[regions.region_id==o.region_id].iloc[0]
    clean=page_clean[o.page]; pred=cv2.imread(str(o.output),0)
    mask=cv2.imread(str(N45/'masks'/o.page/r.region_id.replace(':','_')/f'{o.arm}.png'),0)>0
    if pred is None or pred.shape!=clean.shape: raise RuntimeError(f'Invalid scored output: {o.output}')
    if not mask.any():
        row={**{k:o[k] for k in ['region_id','page','source_group','arm','restorer']},'pixels':0,'mae':np.nan,'ssim_local':np.nan,'ink_precision':np.nan,'ink_recall':np.nan,'ink_f1':np.nan,'edge_f1':np.nan,'false_inserted_ink':0,'missing_ink':0}
    else:
        diff=np.abs(clean.astype(np.float32)-pred.astype(np.float32)); ip,ir,if1=f1(ink(clean),ink(pred),mask); ep,er,ef1=f1(edges(clean),edges(pred),mask)
        _,sm=ssim(clean,pred,data_range=255,full=True)
        row={**{k:o[k] for k in ['region_id','page','source_group','arm','restorer']},'pixels':int(mask.sum()),'mae':float(diff[mask].mean()),'ssim_local':float(sm[mask].mean()),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_f1':ef1,'false_inserted_ink':int((ink(pred)&~ink(clean)&mask).sum()),'missing_ink':int((ink(clean)&~ink(pred)&mask).sum())}
    rows.append(row); done.add(key)
    if len(rows)%100==0: atomic_csv(pd.DataFrame(rows),metric_path)
metrics=pd.DataFrame(rows)
if len(metrics)!=expected or metrics.duplicated(['region_id','arm','restorer']).any(): raise RuntimeError('Metric inventory incomplete or duplicated.')
atomic_csv(metrics,metric_path)
print('SCORING COMPLETE:',len(metrics))

In [ ]:
# Whole-page candidate scoring. Clean truth is used only here, after candidate assembly.
whole_candidates=pd.read_csv(TABLES/'whole_page_candidate_inventory.csv')
whole_metric_rows=[]
for _,row in whole_candidates.iterrows():
    page=row.page; clean=page_clean[page]; obs=page_obs[page]; auth=page_auth[page]
    pred=cv2.imread(str(row.candidate),cv2.IMREAD_GRAYSCALE)
    if pred is None or pred.shape!=clean.shape: raise RuntimeError(f'Invalid whole-page candidate: {row.candidate}')
    diff=np.abs(clean.astype(np.float32)-pred.astype(np.float32))
    _,sm=ssim(clean,pred,data_range=255,full=True)
    ip,ir,if1=f1(ink(clean),ink(pred),auth); ep,er,ef1=f1(edges(clean),edges(pred),auth)
    whole_metric_rows.append({
        'page':page,'source_group':row.source_group,'arm':row.arm,'restorer':row.restorer,
        'authoritative_pixels':int(auth.sum()),'mae_authoritative':float(diff[auth].mean()) if auth.any() else np.nan,
        'ssim_authoritative':float(sm[auth].mean()) if auth.any() else np.nan,
        'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_f1':ef1,
        'false_inserted_ink':int((ink(pred)&~ink(clean)&auth).sum()),
        'missing_ink':int((ink(clean)&~ink(pred)&auth).sum()),
        'outside_mask_drift':int(np.sum(pred[~auth]!=obs[~auth])),
        'changed_vs_observed':int(np.sum(pred!=obs)),
    })
whole_page_metrics=pd.DataFrame(whole_metric_rows)
if len(whole_page_metrics)!=len(whole_candidates): raise RuntimeError('Whole-page metric inventory incomplete.')
atomic_csv(whole_page_metrics,TABLES/'whole_page_candidate_metrics.csv')
atomic_csv(whole_page_metrics.groupby(['arm','restorer'],as_index=False).agg(
    pages=('page','nunique'),mae_authoritative=('mae_authoritative','mean'),ssim_authoritative=('ssim_authoritative','mean'),
    ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),false_inserted_ink=('false_inserted_ink','sum'),
    missing_ink=('missing_ink','sum'),outside_mask_drift=('outside_mask_drift','sum')
),TABLES/'whole_page_candidate_summary.csv')
print('WHOLE-PAGE SCORING COMPLETE:',len(whole_page_metrics))

In [ ]:
# Paired mask effects and oracle-normalised recovery by restorer.
wide=metrics.pivot_table(index=['region_id','page','source_group','restorer'],columns='arm',values=['mae','ink_f1','edge_f1','false_inserted_ink','missing_ink'],aggfunc='first').reset_index();wide.columns=['_'.join([str(x) for x in c if str(x)]) for c in wide.columns]
for metric,higher in [('mae',False),('ink_f1',True),('edge_f1',True),('false_inserted_ink',False),('missing_ink',False)]:
 sign=1 if higher else -1;den=sign*(wide[f'{metric}_M4']-wide[f'{metric}_M0']);num=sign*(wide[f'{metric}_M3']-wide[f'{metric}_M0']);wide[f'{metric}_M3_vs_M0_benefit']=num;wide[f'{metric}_oracle_normalised']=np.where(np.abs(den)>1e-9,num/den,np.nan)
wide.to_csv(TABLES/'oracle_normalised_recovery.csv',index=False)
page=metrics.groupby(['page','source_group','arm','restorer'],as_index=False).agg(regions=('region_id','nunique'),mae=('mae','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),false_inserted_ink=('false_inserted_ink','sum'),missing_ink=('missing_ink','sum'));page.to_csv(TABLES/'results_by_page.csv',index=False)
group=page.groupby(['source_group','arm','restorer'],as_index=False).agg(pages=('page','nunique'),mae=('mae','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),false_inserted_ink=('false_inserted_ink','sum'),missing_ink=('missing_ink','sum'));group.to_csv(TABLES/'results_by_source_group.csv',index=False)

In [ ]:
# Page-cluster bootstrap for M3 vs M0 by restorer.
rng=np.random.default_rng(SEED);boot_rows=[]
for rest,g in page.groupby('restorer'):
 p=g.pivot(index='page',columns='arm',values=['mae','ink_f1','edge_f1'])
 pages=p.index.to_numpy()
 for metric,higher in [('mae',False),('ink_f1',True),('edge_f1',True)]:
  vals=(p[(metric,'M3')]-p[(metric,'M0')])*(1 if higher else -1);samples=[]
  for _ in range(BOOT):samples.append(float(vals.loc[rng.choice(pages,len(pages),replace=True)].mean()))
  boot_rows.append({'restorer':rest,'metric':metric,'benefit_mean':float(vals.mean()),'ci95_low':float(np.quantile(samples,.025)),'ci95_high':float(np.quantile(samples,.975))})
boot=pd.DataFrame(boot_rows);boot.to_csv(TABLES/'paired_bootstrap_M3_vs_M0.csv',index=False)

In [ ]:
# Diagnostic visual queue.
rank=wide.sort_values('edge_f1_M3_vs_M0_benefit',ascending=False)
for kind,data in [('largest_gains',rank.head(50)),('largest_regressions',rank.tail(50))]:
 for i,(_,q) in enumerate(data.iterrows(),1):
  r=regions[regions.region_id==q.region_id].iloc[0];page=q.page;obs=page_obs[page];clean=page_clean[page];m=45;x0=max(0,int(r.x0)-m);x1=min(obs.shape[1],int(r.x1)+m);y0=max(0,int(r.y0)-m);y1=min(obs.shape[0],int(r.y1)+m)
  paths=[OUTPUTS/page/r.region_id.replace(':','_')/f'M0_{q.restorer}.png',OUTPUTS/page/r.region_id.replace(':','_')/f'M3_{q.restorer}.png',OUTPUTS/page/r.region_id.replace(':','_')/f'M4_{q.restorer}.png']
  panels=[obs,cv2.imread(str(paths[0]),0),cv2.imread(str(paths[1]),0),cv2.imread(str(paths[2]),0),clean];titles=['observed','M0','M3','M4','clean diagnostic']
  fig,axes=plt.subplots(1,5,figsize=(15,3))
  for ax,im,t in zip(axes,panels,titles):ax.imshow(im[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=255);ax.set_title(t);ax.axis('off')
  plt.tight_layout();plt.savefig(REVIEW/f'{kind}_{i:03d}_{q.region_id.replace(":","_")}_{q.restorer}.png',dpi=150,bbox_inches='tight');plt.close(fig)

In [ ]:
# Fail-closed development verdict.
learned='R3_learned_unet' if ENABLE_LEARNED_RESTORER else 'R1_biharmonic';b=boot[(boot.restorer==learned)&(boot.metric=='edge_f1')]
positive=bool(len(b) and b.iloc[0].ci95_low>0);stable=bool((group[group.restorer==learned].groupby('source_group').size()>0).all())
verdict={'version':'8.6-notebook46','valid_run':True,'mask_lock_id':lock['lock_id'],'primary_restorer':learned,'M3_edge_f1_bootstrap_positive':positive,'source_groups_reported':stable,'decision':'V8_6_POSITIVE_DEVELOPMENT_FINDING' if positive else 'NO_DEMONSTRATED_EASIER_RECONSTRUCTION','production_authority':False}
(OUT/'v8_6_verdict.json').write_text(json.dumps(verdict,indent=2));print(json.dumps(verdict,indent=2))

In [ ]:
artifacts=['source_record.json','restorer_pre_registration.json','v8_6_verdict.json','tables/paired_region_metrics.csv','tables/oracle_normalised_recovery.csv','tables/results_by_page.csv','tables/results_by_source_group.csv','tables/paired_bootstrap_M3_vs_M0.csv','audit/output_inventory.csv','audit/learned_model_inventory.csv','audit/recovery_audit_before_compute.csv','audit/output_completion.json','audit/page_checkpoint_status.csv','tables/whole_page_candidate_inventory.csv','tables/whole_page_candidate_metrics.csv','tables/whole_page_candidate_summary.csv']
rows=[]
for rel in artifacts:
    p=OUT/rel
    if not p.exists():raise RuntimeError(f'Missing upload artefact {rel}')
    dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst)
    rows.append({'relative_path':rel,'bytes':p.stat().st_size,'sha256':sha(p)})
atomic_csv(pd.DataFrame(rows),UPLOAD/'upload_manifest.csv')
(UPLOAD/'README.txt').write_text('V8.6 Notebook 46R4 fast resumable curated review package. Development evidence only.\n')
(UPLOAD/'upload_file_list.txt').write_text('\n'.join(x['relative_path'] for x in rows))
zip_path=UPLOAD/'notebook46R4_upload_bundle.zip'
tmp=UPLOAD/'notebook46R_upload_bundle.tmp.zip'
with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.rglob('*'):
        if p.is_file() and p not in {zip_path,tmp}:z.write(p,p.relative_to(UPLOAD))
tmp.replace(zip_path)
print('NOTEBOOK 46R4 COMPLETE:',OUT)

In [ ]:
# 46R6 notebook release self-check. This validates the notebook file when available in Colab.
release_record = {
    "notebook_version": NOTEBOOK_VERSION,
    "contract_fingerprint": RUN_CONTRACT_FINGERPRINT,
    "development_only": True,
    "checks": {
        "outside_mask_immutability_required": True,
        "truth_used_only_after_candidate_assembly": True,
        "source_group_separation_required_for_learned_arm": True,
        "checkpoint_reuse_requires_contract_match": True,
        "final_upload_fail_closed": True,
    },
}
release_path = OUT / "46R6_release_record.json" if "OUT" in globals() else Path("46R6_release_record.json")
atomic_write_json(release_path, release_record)
print("Wrote", release_path)
